<a href="https://colab.research.google.com/github/raymondtyaba/case-manager-dashboard/blob/main/project2_handle_missing_values.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project 2: Clean the CDC PLACES Research Variables

**Data-preparation path:** Path A for this step because both diagnosed-diabetes prevalence and food-insecurity prevalence are stored in one CDC PLACES file. No external join is required.

Run every cell from top to bottom. When prompted, upload the CDC PLACES `.xlsx` file.

In [1]:
!pip install -q pandas openpyxl

In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display
from google.colab import files
print("pandas and openpyxl are ready")

pandas and openpyxl are ready


## Upload and load the Excel file

In [3]:
uploaded = files.upload()
print("Upload completed")

Saving CDC_PLACES_County_Data_2025.xlsx to CDC_PLACES_County_Data_2025.xlsx
Upload completed


In [4]:
excel_files = [p for p in Path('/content').glob('*.xlsx') if not p.name.startswith('~$')]
if not excel_files:
    raise FileNotFoundError("No .xlsx file was found. Upload the CDC PLACES Excel file and rerun this cell.")
preferred = [p for p in excel_files if 'cdc' in p.name.lower() or 'place' in p.name.lower()]
file_path = preferred[0] if preferred else excel_files[0]
workbook = pd.ExcelFile(file_path, engine='openpyxl')
df = pd.read_excel(file_path, sheet_name=workbook.sheet_names[0], engine='openpyxl')
df = df.dropna(axis=0, how='all').dropna(axis=1, how='all')
print(f"Loaded: {file_path.name}")
print(f"Worksheet: {workbook.sheet_names[0]}")
print("Original shape:", df.shape)
display(df.head())

Loaded: CDC_PLACES_County_Data_2025.xlsx
Worksheet: in
Original shape: (229298, 22)


,Year,StateAbbr,StateDesc,LocationName,DataSource,Category,Measure,Data_Value_Unit,Data_Value_Type,Data_Value,...,Low_Confidence_Limit,High_Confidence_Limit,TotalPopulation,TotalPop18plus,LocationID,CategoryID,MeasureId,DataValueTypeID,Short_Question_Text,Geolocation
0,2023,AR,Arkansas,Drew,BRFSS,Health Outcomes,Arthritis among adults,%,Crude prevalence,29.9,...,26.6,33.3,16945,13230,5043,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis,POINT (-91.7196579038053 33.5894113647675)
1,2023,AR,Arkansas,Fulton,BRFSS,Health Outcomes,Current asthma among adults,%,Crude prevalence,10.6,...,9.2,11.9,12421,9795,5049,HLTHOUT,CASTHMA,CrdPrv,Current Asthma,POINT (-91.817888079321 36.3816206347765)
2,2023,AR,Arkansas,Howard,BRFSS,Health Outcomes,Arthritis among adults,%,Crude prevalence,31.2,...,27.7,34.8,12533,9311,5061,HLTHOUT,ARTHRITIS,CrdPrv,Arthritis,POINT (-93.9938053308515 34.0889044688856)
3,2023,AR,Arkansas,Miller,BRFSS,Health Outcomes,Stroke among adults,%,Crude prevalence,4.7,...,4.2,5.3,42415,32396,5091,HLTHOUT,STROKE,CrdPrv,Stroke,POINT (-93.8924281761086 33.3123156126392)
4,2023,AR,Arkansas,Ouachita,BRFSS,Disability,Any disability among adults,%,Crude prevalence,42.8,...,37.9,47.8,21793,16948,5103,DISABLT,DISABILITY,CrdPrv,Any Disability,POINT (-92.882040791321 33.593253927504)


## Identify the required columns safely

In [5]:
def find_column(dataframe, exact_names, contains=None):
    lookup = {str(c).strip().lower(): c for c in dataframe.columns}
    for name in exact_names:
        result = lookup.get(name.strip().lower())
        if result is not None:
            return result
    if contains:
        for column in dataframe.columns:
            label = str(column).strip().lower()
            if all(term.lower() in label for term in contains):
                return column
    return None

measure_col = find_column(df, ['Measure'])
value_col = find_column(df, ['Data_Value', 'Data Value'], ['data','value'])
type_col = find_column(df, ['Data_Value_Type', 'Data Value Type'], ['value','type'])
year_col = find_column(df, ['Year'])
location_id_col = find_column(df, ['LocationID', 'Location ID'], ['location','id'])
location_name_col = find_column(df, ['LocationName', 'Location Name'], ['location','name'])
state_col = find_column(df, ['StateAbbr', 'State Abbreviation', 'StateDesc'])

required = {'Measure':measure_col, 'Data_Value':value_col, 'LocationID':location_id_col}
missing = [name for name,col in required.items() if col is None]
if missing:
    raise KeyError("Required columns not found: " + ", ".join(missing) + ". Review df.columns and confirm the correct worksheet.")

for label,col in {'Measure':measure_col,'Value':value_col,'Value type':type_col,'Year':year_col,'Location ID':location_id_col}.items():
    print(f"{label}: {col}")

Measure: Measure
Value: Data_Value
Value type: Data_Value_Type
Year: Year
Location ID: LocationID


## Standardise the fields used to select the two research measures

In [6]:
df_clean = df.copy()

# Text fields used for filtering and matching are standardised.
for column in [measure_col, type_col, location_name_col, state_col]:
    if column is not None:
        df_clean[column] = df_clean[column].astype('string').str.strip()

# Location identifiers are preserved as strings so leading zeroes cannot be lost.
df_clean[location_id_col] = df_clean[location_id_col].astype('string').str.strip().str.replace(r'\.0$', '', regex=True)

# Convert prevalence values to numeric. Percent signs and commas are removed safely.
value_text = df_clean[value_col].astype('string').str.strip().str.replace('%','',regex=False).str.replace(',','',regex=False)
df_clean[value_col] = pd.to_numeric(value_text, errors='coerce')

print("Cleaned source value dtype:", df_clean[value_col].dtype)

Cleaned source value dtype: Float64


## Select the diabetes and food-insecurity records

In [7]:
measure_text = df_clean[measure_col].astype('string')
diabetes_mask = measure_text.str.contains('diabetes', case=False, na=False)
food_mask = measure_text.str.contains('food insecurity', case=False, na=False)

print("Diabetes measure labels:")
print(sorted(df_clean.loc[diabetes_mask, measure_col].dropna().unique().tolist()))
print("Food-insecurity measure labels:")
print(sorted(df_clean.loc[food_mask, measure_col].dropna().unique().tolist()))

if not diabetes_mask.any():
    raise ValueError("No diabetes measure was found. Review the printed Measure values.")
if not food_mask.any():
    raise ValueError("No food-insecurity measure was found. Review the printed Measure values.")

Diabetes measure labels:
['Diagnosed diabetes among adults']
Food-insecurity measure labels:
['Food insecurity in the past 12 months among adults']


## Use one consistent prevalence type and measurement year

In [8]:
selected_mask = diabetes_mask | food_mask

# Prefer age-adjusted prevalence when available for both measures.
selected_value_type = None
if type_col is not None:
    available_types = df_clean.loc[selected_mask, type_col].dropna().astype(str).unique().tolist()
    age_types = [x for x in available_types if 'age-adjusted' in x.lower()]
    if age_types:
        candidate = age_types[0]
        diabetes_has = (diabetes_mask & df_clean[type_col].eq(candidate)).any()
        food_has = (food_mask & df_clean[type_col].eq(candidate)).any()
        if diabetes_has and food_has:
            selected_value_type = candidate
            selected_mask &= df_clean[type_col].eq(candidate)

# Use the latest common year when a year field exists.
selected_year = None
if year_col is not None:
    numeric_year = pd.to_numeric(df_clean[year_col], errors='coerce')
    diabetes_years = set(numeric_year[diabetes_mask & selected_mask].dropna().astype(int))
    food_years = set(numeric_year[food_mask & selected_mask].dropna().astype(int))
    common_years = sorted(diabetes_years & food_years)
    if common_years:
        selected_year = common_years[-1]
        selected_mask &= numeric_year.eq(selected_year)

research_long = df_clean.loc[selected_mask].copy()
print("Selected prevalence type:", selected_value_type if selected_value_type else "All available types")
print("Selected common year:", selected_year if selected_year else "Not filtered")
print("Selected long-format rows:", len(research_long))

Selected prevalence type: Age-adjusted prevalence
Selected common year: 2023
Selected long-format rows: 5258


## Drop exact duplicates and validate county-measure uniqueness

In [9]:
before = research_long.shape
research_long = research_long.drop_duplicates()
after = research_long.shape
print("Before exact deduplication:", before)
print("After exact deduplication:", after)
print("Exact duplicate rows removed:", before[0] - after[0])

research_long['research_variable'] = pd.NA
research_long.loc[research_long[measure_col].astype('string').str.contains('diabetes', case=False, na=False), 'research_variable'] = 'diagnosed_diabetes_prevalence'
research_long.loc[research_long[measure_col].astype('string').str.contains('food insecurity', case=False, na=False), 'research_variable'] = 'food_insecurity_prevalence'

key_cols = [location_id_col, 'research_variable']
duplicate_keys = research_long.duplicated(subset=key_cols, keep=False)
print("Rows with duplicate county-variable keys:", int(duplicate_keys.sum()))

if duplicate_keys.any():
    # Identical county-variable values are safely reduced to one row.
    conflicts = research_long.loc[duplicate_keys].groupby(key_cols)[value_col].nunique(dropna=False)
    true_conflicts = conflicts[conflicts > 1]
    if not true_conflicts.empty:
        raise ValueError("Conflicting values exist for the same county and research variable. Review duplicate records before continuing.")
    research_long = research_long.drop_duplicates(subset=key_cols, keep='first')

Before exact deduplication: (5258, 22)
After exact deduplication: (5258, 22)
Exact duplicate rows removed: 0
Rows with duplicate county-variable keys: 0


## Reshape and rename the two research variables

In [10]:
analysis_df = research_long.pivot(
    index=location_id_col,
    columns='research_variable',
    values=value_col
).reset_index()
analysis_df.columns.name = None

# Add readable geographic fields without changing the two analytical variables.
geo_cols = [c for c in [location_id_col, location_name_col, state_col] if c is not None]
geo = research_long[geo_cols].drop_duplicates(subset=[location_id_col])
analysis_df = geo.merge(analysis_df, on=location_id_col, how='right', validate='one_to_one')

expected = ['diagnosed_diabetes_prevalence','food_insecurity_prevalence']
missing_expected = [c for c in expected if c not in analysis_df.columns]
if missing_expected:
    raise KeyError("Reshaping did not create: " + ", ".join(missing_expected))

for column in expected:
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors='coerce')

print("Analysis-ready shape before missing-value handling:", analysis_df.shape)
print(analysis_df[expected].dtypes)
display(analysis_df.head())

Analysis-ready shape before missing-value handling: (2958, 5)
diagnosed_diabetes_prevalence    Float64
food_insecurity_prevalence       Float64
dtype: object


,LocationID,LocationName,StateAbbr,diagnosed_diabetes_prevalence,food_insecurity_prevalence
0,10001,Kent,DE,12.4,16.6
1,10003,New Castle,DE,10.7,14.6
2,10005,Sussex,DE,9.8,14.5
3,1001,Autauga,AL,11.4,17.0
4,1003,Baldwin,AL,10.4,14.2


## Check ranges and unresolved duplicates

In [11]:
print("Duplicate county IDs:", int(analysis_df[location_id_col].duplicated().sum()))
for column in ['diagnosed_diabetes_prevalence','food_insecurity_prevalence']:
    invalid = analysis_df[column].notna() & ~analysis_df[column].between(0,100)
    print(f"{column}: {int(invalid.sum())} values outside 0-100")
    if invalid.any():
        raise ValueError(f"{column} contains values outside the expected 0-100 percentage range.")

Duplicate county IDs: 0
diagnosed_diabetes_prevalence: 0 values outside 0-100
food_insecurity_prevalence: 0 values outside 0-100


## Confirm the cleaned variables

In [12]:
research_columns = ['diagnosed_diabetes_prevalence','food_insecurity_prevalence']
analysis_df[research_columns].info()
print()
display(analysis_df[research_columns].describe())
print("Missing values before the next missing-data task:")
print(analysis_df[research_columns].isna().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2958 entries, 0 to 2957
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   diagnosed_diabetes_prevalence  2957 non-null   Float64
 1   food_insecurity_prevalence     2300 non-null   Float64
dtypes: Float64(2)
memory usage: 52.1 KB



,diagnosed_diabetes_prevalence,food_insecurity_prevalence
count,2957.0,2300.0
mean,11.167907,17.297652
std,2.329725,6.094196
min,6.0,5.9
25%,9.5,12.9
50%,10.7,15.8
75%,12.5,20.025
max,23.8,46.8


Missing values before the next missing-data task:
diagnosed_diabetes_prevalence      1
food_insecurity_prevalence       658
dtype: int64


## Save the cleaned dataset

In [13]:
output_file = '/content/cdc_places_cleaned_research_variables.xlsx'
analysis_df.to_excel(output_file, index=False, engine='openpyxl')
print("Saved:", output_file)
files.download(output_file)

Saved: /content/cdc_places_cleaned_research_variables.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Handle Missing Values in the Two Research Variables

This section follows the missing-value decision guide for the two numeric research variables in `analysis_df`:

- `diagnosed_diabetes_prevalence`
- `food_insecurity_prevalence`

The rules applied are:

- **0% to under 5% missing:** drop incomplete rows.
- **5% to under 30% missing:** median imputation for the affected numeric variable.
- **30% or more missing:** do not automatically impute; stop and document the limitation for investigation.

## Step 1: Recognise common placeholder codes

In [14]:
import numpy as np

research_columns = [
    'diagnosed_diabetes_prevalence',
    'food_insecurity_prevalence'
]

missing_columns = [column for column in research_columns if column not in analysis_df.columns]
if missing_columns:
    raise KeyError('Missing research columns: ' + ', '.join(missing_columns))

# Ensure both variables are numeric. Text such as N/A is converted to NaN.
for column in research_columns:
    analysis_df[column] = pd.to_numeric(analysis_df[column], errors='coerce')

# Replace clearly invalid sentinel codes only when present.
placeholder_codes = [-999, -9999]
analysis_df[research_columns] = analysis_df[research_columns].replace(placeholder_codes, np.nan)

print('Placeholder-code check completed.')
print(analysis_df[research_columns].dtypes)

Placeholder-code check completed.
diagnosed_diabetes_prevalence    Float64
food_insecurity_prevalence       Float64
dtype: object


## Step 2: Count and calculate missing percentages

In [15]:
rows_before = len(analysis_df)
missing_before = analysis_df[research_columns].isnull().sum()
missing_percentage_before = analysis_df[research_columns].isnull().mean() * 100

missing_summary_before = pd.DataFrame({
    'missing_count': missing_before,
    'missing_percentage': missing_percentage_before.round(2),
    'non_missing_count': analysis_df[research_columns].notnull().sum()
})

print(f'Total county rows before handling missing values: {rows_before:,}')
display(missing_summary_before)

Total county rows before handling missing values: 2,958


,missing_count,missing_percentage,non_missing_count
diagnosed_diabetes_prevalence,1,0.03,2957
food_insecurity_prevalence,658,22.24,2300


## Step 3: Select a method using the decision guide

In [16]:
def choose_missing_method(missing_percentage):
    if missing_percentage < 5:
        return 'drop incomplete rows'
    if missing_percentage < 30:
        return 'median imputation'
    return 'investigate and document; no automatic imputation'

missing_decisions = pd.DataFrame({
    'missing_percentage': missing_percentage_before.round(2),
    'chosen_method': [
        choose_missing_method(missing_percentage_before[column])
        for column in research_columns
    ]
})

display(missing_decisions)

high_missing_columns = [
    column for column in research_columns
    if missing_percentage_before[column] >= 30
]

if high_missing_columns:
    print('Caution: 30% or more missing in:', high_missing_columns)
    print('These variables will not be automatically imputed.')
else:
    print('Neither research variable has 30% or more missing data.')

,missing_percentage,chosen_method
diagnosed_diabetes_prevalence,0.03,drop incomplete rows
food_insecurity_prevalence,22.24,median imputation


Neither research variable has 30% or more missing data.


## Step 4: Apply the chosen method

In [17]:
analysis_ready_df = analysis_df.copy()
method_log = []

# Stop before applying a potentially misleading fix when missingness is structural.
if high_missing_columns:
    raise ValueError(
        'At least one research variable has 30% or more missing data. '
        'Investigate the dataset documentation and record this as a limitation before continuing: '
        + ', '.join(high_missing_columns)
    )

# Median-impute only variables in the 5% to under 30% range.
for column in research_columns:
    percentage = missing_percentage_before[column]
    if 5 <= percentage < 30:
        median_value = analysis_ready_df[column].median()
        if pd.isna(median_value):
            raise ValueError(f'Cannot calculate a median for {column}.')
        analysis_ready_df[column] = analysis_ready_df[column].fillna(median_value)
        method_log.append(
            f'{column}: median imputation using {median_value:.2f} '
            f'({percentage:.2f}% missing before treatment)'
        )

# For correlation, both variables must be observed in each retained county.
# If either variable is under 5% missing, complete-case deletion implements that decision.
low_missing_columns = [
    column for column in research_columns
    if 0 < missing_percentage_before[column] < 5
]

if low_missing_columns:
    before_drop = len(analysis_ready_df)
    analysis_ready_df = analysis_ready_df.dropna(subset=low_missing_columns).copy()
    rows_dropped = before_drop - len(analysis_ready_df)
    method_log.append(
        'Complete-case deletion for variables under 5% missing: '
        + ', '.join(low_missing_columns)
        + f'; {rows_dropped:,} county rows removed'
    )

if not method_log:
    method_log.append('No missing-value treatment was required because both variables were complete.')

print('Methods applied:')
for entry in method_log:
    print('-', entry)

Methods applied:
- food_insecurity_prevalence: median imputation using 15.80 (22.24% missing before treatment)
- Complete-case deletion for variables under 5% missing: diagnosed_diabetes_prevalence; 1 county rows removed


## Step 5: Confirm the result

In [18]:
missing_after = analysis_ready_df[research_columns].isnull().sum()
missing_percentage_after = analysis_ready_df[research_columns].isnull().mean() * 100

missing_summary_after = pd.DataFrame({
    'missing_count_after': missing_after,
    'missing_percentage_after': missing_percentage_after.round(2),
    'non_missing_count_after': analysis_ready_df[research_columns].notnull().sum()
})

print(f'County rows before treatment: {rows_before:,}')
print(f'County rows after treatment: {len(analysis_ready_df):,}')
display(missing_summary_after)

if analysis_ready_df[research_columns].isnull().any().any():
    raise ValueError('Missing values remain in the two research variables. Review the treatment cells.')

if len(analysis_ready_df) < 50:
    raise ValueError('Fewer than 50 complete county records remain, so the planned analysis needs review.')

print('Validation passed: both research variables contain 0 missing values.')
print(f'Usable matched counties: {len(analysis_ready_df):,}')

County rows before treatment: 2,958
County rows after treatment: 2,957


,missing_count_after,missing_percentage_after,non_missing_count_after
diagnosed_diabetes_prevalence,0,0.0,2957
food_insecurity_prevalence,0,0.0,2957


Validation passed: both research variables contain 0 missing values.
Usable matched counties: 2,957


## Compare descriptive statistics before and after treatment

In [19]:
print('Before missing-value treatment:')
display(analysis_df[research_columns].describe())

print('After missing-value treatment:')
display(analysis_ready_df[research_columns].describe())

Before missing-value treatment:


,diagnosed_diabetes_prevalence,food_insecurity_prevalence
count,2957.0,2300.0
mean,11.167907,17.297652
std,2.329725,6.094196
min,6.0,5.9
25%,9.5,12.9
50%,10.7,15.8
75%,12.5,20.025
max,23.8,46.8


After missing-value treatment:


,diagnosed_diabetes_prevalence,food_insecurity_prevalence
count,2957.0,2957.0
mean,11.167907,16.964897
std,2.329725,5.4104
min,6.0,5.9
25%,9.5,13.7
50%,10.7,15.8
75%,12.5,18.4
max,23.8,46.8


## Create the final written record

In [20]:
def treatment_sentence(column):
    percentage = missing_percentage_before[column]
    count = int(missing_before[column])
    method = choose_missing_method(percentage)
    return (
        f'{column} had {count:,} missing values ({percentage:.2f}%). '
        f'The selected approach was {method}.'
    )

print(treatment_sentence('diagnosed_diabetes_prevalence'))
print(treatment_sentence('food_insecurity_prevalence'))
print(
    f'After treatment, {len(analysis_ready_df):,} county records remained and both '
    'research variables had zero missing values.'
)

diagnosed_diabetes_prevalence had 1 missing values (0.03%). The selected approach was drop incomplete rows.
food_insecurity_prevalence had 658 missing values (22.24%). The selected approach was median imputation.
After treatment, 2,957 county records remained and both research variables had zero missing values.


## Save and download the analysis-ready dataset

In [21]:
final_output = '/content/cdc_places_analysis_ready.xlsx'
analysis_ready_df.to_excel(final_output, index=False, engine='openpyxl')
print('Saved:', final_output)
files.download(final_output)

Saved: /content/cdc_places_analysis_ready.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>